# LeetCode 3823 — Reverse Letters Then Special Characters in a String bằng Python

Link đề: https://leetcode.com/problems/reverse-letters-then-special-characters-in-a-string/ · Độ khó: **Easy** · Chủ đề: String, Two Pointers, Simulation

Skeleton LeetCode:

```python
class Solution:
    def reverseByType(self, s: str) -> str:
        ...
```

> Ghi chú: tên bài, đề, ví dụ, constraints và chữ ký hàm đã được đối chiếu qua WebSearch / leetcode.doocs.org. Một nguồn xếp bài này ở mức khó khác; trang doocs ghi **Easy** (Biweekly Contest 175, câu 1) nên lưu ở DSA/Easy.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chuỗi `s` gồm **chữ cái thường** và **ký tự đặc biệt** `!@#$%^&*()`; đảo các chữ cái **với nhau** (trả về đúng các vị trí chữ cái) rồi đảo các ký tự đặc biệt **với nhau** (trả về đúng các vị trí ký tự đặc biệt).
2. Nhận ra hai nhóm ký tự là **độc lập**: vị trí của chúng được giữ nguyên, chỉ nội dung trong mỗi nhóm bị đảo.
3. Biết phân loại ký tự bằng `ch.islower()` / `'a' <= ch <= 'z'`, tách thành hai list, và dùng `reversed`, `iter`, `next` để điền ngược lại.
4. Viết 5 cách khác nhau (đối xứng theo thứ hạng, hai list + iterator, hai con trỏ cho từng nhóm, danh sách vị trí, ngăn xếp) và đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: đảo cả chuỗi, chỉ đảo một nhóm, ghép hai nhóm liền nhau làm mất vị trí, và hai con trỏ không bỏ qua ký tự khác loại.

## 2. Đề bài & ví dụ

Cho chuỗi `s` gồm các chữ cái tiếng Anh thường và các ký tự đặc biệt. Thực hiện lần lượt hai bước:

1. **Đảo các chữ cái** với nhau: các chữ cái được đặt lại vào chính các vị trí đang chứa chữ cái, nhưng theo thứ tự đảo ngược.
2. **Đảo các ký tự đặc biệt** với nhau: tương tự, đặt vào các vị trí đang chứa ký tự đặc biệt theo thứ tự đảo ngược.

Trả về chuỗi kết quả.

| Ví dụ | s | Output | Giải thích |
|---|---|---|---|
| 1 | `")ebc#da@f("` | `"(fad@cb#e)"` | Chữ cái `e,b,c,d,a,f` đảo thành `f,a,d,c,b,e`; ký tự đặc biệt `),#,@,(` đảo thành `(,@,#,)` |
| 2 | `"z"` | `"z"` | Chỉ một chữ cái, đảo không đổi, không có ký tự đặc biệt |
| 3 | `"!@#$%^&*()"` | `")(*&^%$#@!"` | Không có chữ cái; toàn bộ là ký tự đặc biệt nên đảo cả chuỗi |

Constraints: `1 <= s.length <= 100`; `s` chỉ gồm chữ cái tiếng Anh thường và các ký tự `!@#$%^&*()`.

In [1]:
for s in [")ebc#da@f(", "z", "!@#$%^&*()"]:
    letters = [c for c in s if c.islower()]
    specials = [c for c in s if not c.islower()]
    print(f"s={s!r}  chữ cái={letters}  đặc biệt={specials}")

s=')ebc#da@f('  chữ cái=['e', 'b', 'c', 'd', 'a', 'f']  đặc biệt=[')', '#', '@', '(']
s='z'  chữ cái=['z']  đặc biệt=[]
s='!@#$%^&*()'  chữ cái=[]  đặc biệt=['!', '@', '#', '$', '%', '^', '&', '*', '(', ')']


## 3. Kiến thức nền

### 3.1. Phân loại ký tự

`ch.islower()` là `True` nếu `ch` là chữ cái thường. Với bài này mọi ký tự không phải chữ cái thường đều là ký tự đặc biệt (`!@#$%^&*()`). Có thể thay bằng so sánh `'a' <= ch <= 'z'`. (Hàm `ch.isalpha()` cũng hợp ở đây vì không có chữ in hoa hay chữ số.)

In [2]:
for ch in "a#z(":
    print(repr(ch), ch.islower(), 'a' <= ch <= 'z', ch.isalpha())

'a' True True True
'#' False False False
'z' True True True
'(' False False False


### 3.2. Lọc thành hai list

List comprehension `[c for c in s if điều_kiện]` tạo list mới. Ta tách `letters` và `specials` **theo đúng thứ tự xuất hiện**.

In [3]:
s = ")ebc#da@f("
letters = [c for c in s if c.islower()]
specials = [c for c in s if not c.islower()]
print(letters, specials)
print(len(letters) + len(specials) == len(s))

['e', 'b', 'c', 'd', 'a', 'f'] [')', '#', '@', '(']
True


### 3.3. `reversed`, `iter` và `next`

`reversed(list)` trả iterator đi từ cuối về đầu. `iter(x)` tạo iterator; mỗi lần `next(it)` lấy phần tử kế tiếp và **ghi nhớ vị trí**. Hợp để điền lần lượt các phần tử đã đảo vào các vị trí cần thiết.

In [4]:
letters = ['e', 'b', 'c', 'd', 'a', 'f']
it = iter(reversed(letters))
print(next(it), next(it), next(it))
it2 = iter([1, 2])
print(next(it2), next(it2), next(it2, "hết"))

f a d
1 2 hết


### 3.4. Hai con trỏ và bỏ qua phần tử khác loại

Mẫu "đảo tại chỗ": `i` từ đầu, `j` từ cuối; hoán đổi khi cả hai trỏ vào phần tử **cùng loại**. Với một nhóm (ví dụ chữ cái), `while i < j` rồi: nếu `s[i]` không thuộc nhóm thì `i += 1`; nếu `s[j]` không thuộc nhóm thì `j -= 1`; ngược lại hoán đổi và thu hẹp.

In [5]:
chars = list("a#b")
i, j = 0, len(chars) - 1
while i < j:
    if not chars[i].islower():
        i += 1
    elif not chars[j].islower():
        j -= 1
    else:
        chars[i], chars[j] = chars[j], chars[i]
        i += 1
        j -= 1
print(''.join(chars))

b#a


### 3.5. Chuỗi bất biến và `''.join`

`str` không sửa tại chỗ được. Ta chuyển sang `list`, sửa theo vị trí, rồi `''.join(list)` để ra chuỗi kết quả.

In [6]:
chars = list("ab#")
chars[0], chars[1] = chars[1], chars[0]
print(chars, ''.join(chars))
try:
    "ab#"[0] = "x"
except TypeError as e:
    print("TypeError:", e)

['b', 'a', '#'] ba#
TypeError: 'str' object does not support item assignment


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Hãy tưởng tượng chuỗi là một dãy ô, mỗi ô có **màu**: ô "chữ cái" và ô "đặc biệt". Màu của mỗi ô **không đổi**; chỉ có nội dung bên trong các ô cùng màu bị đảo thứ tự với nhau.

Tính tay `s = ")ebc#da@f("`:

| vị trí | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |
|---|---|---|---|---|---|---|---|---|---|---|
| ký tự | `)` | `e` | `b` | `c` | `#` | `d` | `a` | `@` | `f` | `(` |
| loại | Đ | C | C | C | Đ | C | C | Đ | C | Đ |

- Chữ cái (theo thứ tự): `e b c d a f` -> đảo: `f a d c b e`, đặt vào các vị trí `1,2,3,5,6,8`.
- Đặc biệt: `) # @ (` -> đảo: `( @ # )`, đặt vào các vị trí `0,4,7,9`.
- Ghép: `( f a d @ c b # e )` = `"(fad@cb#e)"`.

Hai bước của đề (đảo chữ cái rồi đảo ký tự đặc biệt) **không ảnh hưởng nhau** vì chúng ghi vào các tập vị trí rời nhau, nên thứ tự hai bước không quan trọng.

### Bảng tính tay

| s | chữ cái | đảo | đặc biệt | đảo | kết quả |
|---|---|---|---|---|---|
| `")ebc#da@f("` | e b c d a f | f a d c b e | ) # @ ( | ( @ # ) | `(fad@cb#e)` |
| `"z"` | z | z | (không) | | `z` |
| `"!@#$%^&*()"` | (không) | | ! @ # $ % ^ & * ( ) | ) ( * & ^ % $ # @ ! | `)(*&^%$#@!` |
| `"a!b"` | a b | b a | ! | ! | `b!a` |

In [7]:
for s in [")ebc#da@f(", "z", "!@#$%^&*()", "a!b"]:
    L = [c for c in s if c.islower()]
    S = [c for c in s if not c.islower()]
    li, si = iter(L[::-1]), iter(S[::-1])
    print(repr(s), L, S, "->", repr(''.join(next(li) if c.islower() else next(si) for c in s)))

')ebc#da@f(' ['e', 'b', 'c', 'd', 'a', 'f'] [')', '#', '@', '('] -> '(fad@cb#e)'
'z' ['z'] [] -> 'z'
'!@#$%^&*()' [] ['!', '@', '#', '$', '%', '^', '&', '*', '(', ')'] -> ')(*&^%$#@!'
'a!b' ['a', 'b'] ['!'] -> 'b!a'


## 5. Cách 1 — Brute Force / baseline

Với **mỗi vị trí** `i`: đếm `t` = số ký tự **cùng loại** đứng trước `i` (thứ hạng của nó trong nhóm); gọi `m` là tổng số ký tự cùng loại; ký tự kết quả tại `i` là ký tự cùng loại có thứ hạng `m - 1 - t` (đối xứng), tìm bằng cách quét lại chuỗi. Không dùng list phụ, chỉ đếm và quét.

Nút thắt: mỗi vị trí quét lại cả chuỗi nên tổng `O(n²)`.

In [8]:
class SolutionBrute:
    def reverseByType(self, s):
        def same(a, b):
            return a.islower() == b.islower()
        res = []
        for i, ch in enumerate(s):
            t = sum(1 for j in range(i) if same(s[j], ch))          # thứ hạng trong nhóm
            m = sum(1 for c in s if same(c, ch))                    # kích thước nhóm
            target = m - 1 - t                                      # thứ hạng đối xứng
            seen = 0
            for c in s:                                             # tìm ký tự cùng loại có thứ hạng target
                if same(c, ch):
                    if seen == target:
                        res.append(c)
                        break
                    seen += 1
        return ''.join(res)

print(SolutionBrute().reverseByType(")ebc#da@f("), SolutionBrute().reverseByType("z"), SolutionBrute().reverseByType("!@#$%^&*()"))

(fad@cb#e) z )(*&^%$#@!


**Độ phức tạp:** Time `O(n²)`, Space `O(n)` (kết quả).

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** tách thành **hai dãy độc lập**. Lấy dãy chữ cái `L` và dãy đặc biệt `S` theo thứ tự xuất hiện, đảo từng dãy, rồi **đi qua chuỗi gốc một lần**: tại mỗi vị trí, nếu nó là chữ cái thì lấy phần tử kế tiếp của `L` đã đảo, ngược lại lấy từ `S` đã đảo.

**Quy tắc "nếu… thì…":**

- Nếu vị trí `i` là chữ cái -> kết quả là chữ cái kế tiếp trong `reversed(L)`.
- Nếu vị trí `i` là ký tự đặc biệt -> kết quả là ký tự kế tiếp trong `reversed(S)`.
- Loại của mỗi vị trí **không đổi**; số chữ cái và số ký tự đặc biệt không đổi.
- Chuỗi toàn một loại thì kết quả là `s[::-1]`.

**Invariant:** sau khi điền `i` vị trí đầu, các iterator đã dùng đúng số phần tử bằng số chữ cái/ký tự đặc biệt trong `s[:i]`.

In [9]:
# Kiểm chứng: loại tại mỗi vị trí không đổi, và mỗi nhóm bị đảo
import random
random.seed(3823)
SPECIAL = "!@#$%^&*()"
for _ in range(2000):
    n = random.randint(1, 40)
    s = ''.join(random.choice("abcxyz" + SPECIAL) for _ in range(n))
    L = [c for c in s if c.islower()]
    S = [c for c in s if not c.islower()]
    li, si = iter(L[::-1]), iter(S[::-1])
    r = ''.join(next(li) if c.islower() else next(si) for c in s)
    assert [c.islower() for c in r] == [c.islower() for c in s]
    assert [c for c in r if c.islower()] == L[::-1] and [c for c in r if not c.islower()] == S[::-1]
print("loại mỗi vị trí không đổi; mỗi nhóm đảo đúng")

loại mỗi vị trí không đổi; mỗi nhóm đảo đúng


## 7. Các cách tiếp theo

### Cách 2 — Hai list + iterator

Tách `letters`, `specials`, đảo bằng `reversed`, tạo iterator rồi dựng kết quả bằng `next`. Rõ ràng, `O(n)`.

In [10]:
class SolutionIterators:
    def reverseByType(self, s):
        letters = iter(reversed([c for c in s if c.islower()]))
        specials = iter(reversed([c for c in s if not c.islower()]))
        return ''.join(next(letters) if c.islower() else next(specials) for c in s)

print(SolutionIterators().reverseByType(")ebc#da@f("), SolutionIterators().reverseByType("z"), SolutionIterators().reverseByType("!@#$%^&*()"))

(fad@cb#e) z )(*&^%$#@!


**Độ phức tạp:** Time `O(n)`, Space `O(n)`.

### Cách 3 — Hai con trỏ cho từng nhóm (đảo tại chỗ)

Viết một hàm phụ `reverse_group(chars, pred)` đảo các phần tử thỏa `pred` bằng hai con trỏ (bỏ qua phần tử không thỏa); gọi hai lần: một lần cho chữ cái, một lần cho ký tự đặc biệt. Đây đúng là hai bước của đề.

In [11]:
class SolutionTwoPointers:
    def reverseByType(self, s):
        chars = list(s)

        def reverse_group(is_member):
            i, j = 0, len(chars) - 1
            while i < j:
                if not is_member(chars[i]):
                    i += 1
                elif not is_member(chars[j]):
                    j -= 1
                else:
                    chars[i], chars[j] = chars[j], chars[i]
                    i += 1
                    j -= 1

        reverse_group(lambda c: c.islower())          # bước 1: chữ cái
        reverse_group(lambda c: not c.islower())      # bước 2: ký tự đặc biệt
        return ''.join(chars)

print(SolutionTwoPointers().reverseByType(")ebc#da@f("), SolutionTwoPointers().reverseByType("z"), SolutionTwoPointers().reverseByType("!@#$%^&*()"))

(fad@cb#e) z )(*&^%$#@!


**Độ phức tạp:** Time `O(n)` (mỗi con trỏ đi tối đa `n` bước cho mỗi nhóm), Space `O(n)` (list ký tự).

### Cách 4 — Danh sách vị trí

Ghi lại **vị trí** của mỗi nhóm: `pos_letters`, `pos_specials`. Dãy ký tự của nhóm ứng với các vị trí đó; điền ký tự đảo vào đúng các vị trí. Rất trực quan với hình ảnh "ô có màu".

In [12]:
class SolutionPositions:
    def reverseByType(self, s):
        res = list(s)
        for is_letter in (True, False):
            pos = [i for i, c in enumerate(s) if c.islower() == is_letter]
            vals = [s[i] for i in pos]
            for p, v in zip(pos, reversed(vals)):
                res[p] = v
        return ''.join(res)

print(SolutionPositions().reverseByType(")ebc#da@f("), SolutionPositions().reverseByType("z"), SolutionPositions().reverseByType("!@#$%^&*()"))

(fad@cb#e) z )(*&^%$#@!


**Độ phức tạp:** Time `O(n)`, Space `O(n)`.

### Cách 5 — Ngăn xếp cho từng nhóm

Đẩy chữ cái vào `stack_L`, ký tự đặc biệt vào `stack_S`. Duyệt lại chuỗi gốc: tại vị trí chữ cái thì `stack_L.pop()`, tại vị trí đặc biệt thì `stack_S.pop()`. LIFO tự động cho thứ tự đảo.

In [13]:
class SolutionStack:
    def reverseByType(self, s):
        stack_l, stack_s = [], []
        for c in s:
            (stack_l if c.islower() else stack_s).append(c)
        out = []
        for c in s:
            out.append(stack_l.pop() if c.islower() else stack_s.pop())
        return ''.join(out)

print(SolutionStack().reverseByType(")ebc#da@f("), SolutionStack().reverseByType("z"), SolutionStack().reverseByType("!@#$%^&*()"))

(fad@cb#e) z )(*&^%$#@!


**Độ phức tạp:** Time `O(n)`, Space `O(n)`.

## 8. TRACE từng bước

Trace Cách 2 (iterator) với `s = "a#b!c"`:

- `letters = [a, b, c]` -> đảo `c, b, a`; `specials = [#, !]` -> đảo `!, #`.

| vị trí | ký tự gốc | loại | lấy từ | kết quả tại vị trí |
|---|---|---|---|---|
| 0 | a | chữ cái | `c` | c |
| 1 | # | đặc biệt | `!` | ! |
| 2 | b | chữ cái | `b` | b |
| 3 | ! | đặc biệt | `#` | # |
| 4 | c | chữ cái | `a` | a |

Kết quả `"c!b#a"`.

In [14]:
def trace_iterators(s):
    L = [c for c in s if c.islower()]
    S = [c for c in s if not c.islower()]
    print("chữ cái:", L, "-> đảo:", L[::-1])
    print("đặc biệt:", S, "-> đảo:", S[::-1])
    li, si = iter(L[::-1]), iter(S[::-1])
    out = []
    for i, c in enumerate(s):
        if c.islower():
            v = next(li)
            print(f"  vị trí {i}: {c!r} là chữ cái   -> lấy {v!r} từ dãy chữ cái đã đảo")
        else:
            v = next(si)
            print(f"  vị trí {i}: {c!r} là đặc biệt -> lấy {v!r} từ dãy đặc biệt đã đảo")
        out.append(v)
    return ''.join(out)

assert trace_iterators("a#b!c") == "c!b#a"
assert trace_iterators(")ebc#da@f(") == "(fad@cb#e)"

chữ cái: ['a', 'b', 'c'] -> đảo: ['c', 'b', 'a']
đặc biệt: ['#', '!'] -> đảo: ['!', '#']
  vị trí 0: 'a' là chữ cái   -> lấy 'c' từ dãy chữ cái đã đảo
  vị trí 1: '#' là đặc biệt -> lấy '!' từ dãy đặc biệt đã đảo
  vị trí 2: 'b' là chữ cái   -> lấy 'b' từ dãy chữ cái đã đảo
  vị trí 3: '!' là đặc biệt -> lấy '#' từ dãy đặc biệt đã đảo
  vị trí 4: 'c' là chữ cái   -> lấy 'a' từ dãy chữ cái đã đảo
chữ cái: ['e', 'b', 'c', 'd', 'a', 'f'] -> đảo: ['f', 'a', 'd', 'c', 'b', 'e']
đặc biệt: [')', '#', '@', '('] -> đảo: ['(', '@', '#', ')']
  vị trí 0: ')' là đặc biệt -> lấy '(' từ dãy đặc biệt đã đảo
  vị trí 1: 'e' là chữ cái   -> lấy 'f' từ dãy chữ cái đã đảo
  vị trí 2: 'b' là chữ cái   -> lấy 'a' từ dãy chữ cái đã đảo
  vị trí 3: 'c' là chữ cái   -> lấy 'd' từ dãy chữ cái đã đảo
  vị trí 4: '#' là đặc biệt -> lấy '@' từ dãy đặc biệt đã đảo
  vị trí 5: 'd' là chữ cái   -> lấy 'c' từ dãy chữ cái đã đảo
  vị trí 6: 'a' là chữ cái   -> lấy 'b' từ dãy chữ cái đã đảo
  vị trí 7: '@' là đặc biệt 

In [15]:
def trace_two_pointers_group(s, name, is_member):
    chars = list(s)
    i, j = 0, len(chars) - 1
    print(f"nhóm {name}: bắt đầu {''.join(chars)!r}")
    while i < j:
        if not is_member(chars[i]):
            print(f"  i={i} ({chars[i]!r}) khác nhóm -> i += 1")
            i += 1
        elif not is_member(chars[j]):
            print(f"  j={j} ({chars[j]!r}) khác nhóm -> j -= 1")
            j -= 1
        else:
            chars[i], chars[j] = chars[j], chars[i]
            print(f"  hoán đổi {i} và {j} -> {''.join(chars)!r}")
            i += 1
            j -= 1
    return ''.join(chars)

step1 = trace_two_pointers_group("a#b!c", "chữ cái", lambda c: c.islower())
step2 = trace_two_pointers_group(step1, "đặc biệt", lambda c: not c.islower())
assert step2 == "c!b#a"

nhóm chữ cái: bắt đầu 'a#b!c'
  hoán đổi 0 và 4 -> 'c#b!a'
  i=1 ('#') khác nhóm -> i += 1
  j=3 ('!') khác nhóm -> j -= 1
nhóm đặc biệt: bắt đầu 'c#b!a'
  i=0 ('c') khác nhóm -> i += 1
  j=4 ('a') khác nhóm -> j -= 1
  hoán đổi 1 và 3 -> 'c!b#a'


## 9. Code nộp LeetCode

In [16]:
from typing import List, Optional

class Solution:
    def reverseByType(self, s: str) -> str:
        letters = iter(reversed([c for c in s if c.islower()]))
        specials = iter(reversed([c for c in s if not c.islower()]))
        return ''.join(next(letters) if c.islower() else next(specials) for c in s)

## 10. Vì sao thuật toán đúng

- **Hai nhóm không chồng nhau:** mỗi ký tự thuộc đúng một nhóm (chữ cái thường hoặc đặc biệt). Hai bước của đề ghi vào hai tập vị trí rời nhau, nên kết quả không phụ thuộc thứ tự thực hiện hai bước.
- **Đảo trong nhóm đúng định nghĩa:** nếu nhóm có `m` ký tự ở các vị trí `p0 < p1 < ... < p(m-1)` với giá trị `v0..v(m-1)` thì sau khi đảo, vị trí `pt` nhận giá trị `v(m-1-t)`. `reversed(L)` rồi lấy tuần tự bằng `next` cho đúng gán này (vì ta duyệt `s` theo thứ tự tăng của vị trí).
- **Không lấy thiếu/thừa:** số lần `next(letters)` bằng số vị trí chữ cái `= len(L)`, và số lần `next(specials)` bằng `len(S)`, nên iterator không bao giờ cạn sớm.
- **Hai con trỏ:** bỏ qua phần tử khác nhóm rồi hoán đổi hai phần tử nhóm ở hai đầu; lặp lại đến khi gặp nhau, tương đương đảo dãy con của nhóm tại chỗ.
- **Biên:** chuỗi một nhóm cho `s[::-1]`; chuỗi một ký tự không đổi.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| Brute force (thứ hạng đối xứng) | `O(n²)` | `O(n)` |
| Hai list + iterator | `O(n)` | `O(n)` |
| Hai con trỏ cho từng nhóm | `O(n)` | `O(n)` |
| Danh sách vị trí | `O(n)` | `O(n)` |
| Ngăn xếp | `O(n)` | `O(n)` |

`n = len(s) <= 100`. Chuỗi Python bất biến nên mọi cách đều cần ít nhất `O(n)` bộ nhớ để dựng chuỗi mới; không thể `O(1)` thật sự. Mọi cách đều tức thì với `n <= 100`.

## 12. Bộ test

Ví dụ của đề, các biên (một ký tự, chỉ chữ cái, chỉ ký tự đặc biệt, xen kẽ, ký tự đặc biệt ở hai đầu, độ dài `100`) và so sánh **tất cả cách** trên input ngẫu nhiên hợp lệ (chỉ dùng chữ cái thường và `!@#$%^&*()`).

In [17]:
impls = [SolutionBrute(), SolutionIterators(), SolutionTwoPointers(), SolutionPositions(), SolutionStack(), Solution()]

cases = [
    (")ebc#da@f(", "(fad@cb#e)"), ("z", "z"), ("!@#$%^&*()", ")(*&^%$#@!"),
    ("a", "a"), ("!", "!"), ("ab", "ba"), ("!(", "(!"), ("a!b", "b!a"), ("!a!", "!a!"), ("ab!", "ba!"),
    ("a#b!c", "c!b#a"), ("abc", "cba"), ("(ab)", ")ba("), ("a" * 100, "a" * 100), ("abcdefghij" * 10, ("abcdefghij" * 10)[::-1]),
    ("a!" * 50, "a!" * 50),
]
for s, exp in cases:
    assert 1 <= len(s) <= 100 and all(c.islower() or c in "!@#$%^&*()" for c in s)
    for impl in impls:
        got = impl.reverseByType(s)
        assert got == exp, (type(impl).__name__, s, got, exp)
print("ví dụ + edge OK:", len(cases), "ca")

ví dụ + edge OK: 16 ca


In [18]:
import random
random.seed(38230)
SPECIAL = "!@#$%^&*()"
for _ in range(1000):
    n = random.randint(1, 100)
    alpha = random.choice([SPECIAL, "abc", "abc" + SPECIAL, "abcdefghijklmnopqrstuvwxyz" + SPECIAL, "a" + "!"])
    s = ''.join(random.choice(alpha) for _ in range(n))
    L = [c for c in s if c.islower()]
    S = [c for c in s if not c.islower()]
    li, si = iter(L[::-1]), iter(S[::-1])
    exp = ''.join(next(li) if c.islower() else next(si) for c in s)
    outs = [impl.reverseByType(s) for impl in impls]
    assert all(o == exp for o in outs), (s, outs, exp)
    assert len(exp) == n and sorted(exp) == sorted(s)
print("1000 test ngẫu nhiên (seed=38230): 6 cách giống nhau")

1000 test ngẫu nhiên (seed=38230): 6 cách giống nhau


In [19]:
# Tính chất: loại ở mỗi vị trí không đổi; áp dụng hai lần trả về chuỗi gốc
for _ in range(400):
    n = random.randint(1, 100)
    s = ''.join(random.choice("abcxyz" + SPECIAL) for _ in range(n))
    r = Solution().reverseByType(s)
    assert [c.islower() for c in r] == [c.islower() for c in s]
    assert Solution().reverseByType(r) == s
    if all(c.islower() for c in s) or not any(c.islower() for c in s):
        assert r == s[::-1]
print("tính chất bảo toàn loại và đối hợp (involution) OK")

tính chất bảo toàn loại và đối hợp (involution) OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: đảo cả chuỗi

`s[::-1]` đảo cả hai nhóm **cùng lúc**, làm các chữ cái và ký tự đặc biệt đổi chỗ cho nhau (loại ở mỗi vị trí bị đổi).

In [20]:
def wrong_reverse_all(s):
    return s[::-1]                                # SAI

s = ")ebc#da@f("
print(wrong_reverse_all(s), "!=", Solution().reverseByType(s))
assert wrong_reverse_all(s) == "(f@ad#cbe)" != "(fad@cb#e)"

(f@ad#cbe) != (fad@cb#e)


### Lỗi 2: chỉ đảo một nhóm

Quên rằng đề yêu cầu đảo **cả hai** nhóm; đảo mỗi nhóm chữ cái thì các ký tự đặc biệt vẫn ở thứ tự cũ.

In [21]:
def wrong_letters_only(s):
    L = iter(reversed([c for c in s if c.islower()]))
    return ''.join(next(L) if c.islower() else c for c in s)      # SAI: không đảo ký tự đặc biệt

s = ")ebc#da@f("
print(wrong_letters_only(s), "!=", Solution().reverseByType(s))
assert wrong_letters_only(s) == ")fad#cb@e(" != "(fad@cb#e)"
assert wrong_letters_only("abc") == Solution().reverseByType("abc")       # đúng khi không có ký tự đặc biệt

)fad#cb@e( != (fad@cb#e)


### Lỗi 3: ghép hai nhóm liền nhau (mất vị trí)

Đảo từng nhóm rồi nối `letters_reversed + specials_reversed` làm mất thông tin **vị trí xen kẽ** của hai nhóm.

In [22]:
def wrong_concat_groups(s):
    L = [c for c in s if c.islower()][::-1]
    S = [c for c in s if not c.islower()][::-1]
    return ''.join(L + S)                                         # SAI: các vị trí bị dồn lại

s = ")ebc#da@f("
print(wrong_concat_groups(s), "!=", Solution().reverseByType(s))
assert wrong_concat_groups(s) == "fadcbe(@#)" != "(fad@cb#e)"

fadcbe(@#) != (fad@cb#e)


### Lỗi 4: hai con trỏ không bỏ qua ký tự khác loại

Nếu hoán đổi `chars[i]` và `chars[j]` mà không kiểm tra cùng loại thì trộn lẫn chữ cái và ký tự đặc biệt (kết quả giống đảo cả chuỗi).

In [23]:
def wrong_no_skip(s):
    chars = list(s)
    i, j = 0, len(chars) - 1
    while i < j:
        chars[i], chars[j] = chars[j], chars[i]                   # SAI: không kiểm tra cùng loại
        i += 1
        j -= 1
    return ''.join(chars)

s = ")ebc#da@f("
print(wrong_no_skip(s), "!=", Solution().reverseByType(s))
assert wrong_no_skip(s) == s[::-1] != Solution().reverseByType(s)

(f@ad#cbe) != (fad@cb#e)


### Lỗi 5: hai con trỏ chỉ bỏ qua đúng một bước

Nếu dùng hai lệnh `if` độc lập (bỏ qua một lần ở mỗi phía) rồi **hoán đổi luôn**, khi vừa nhảy qua một ký tự khác loại ta lại hoán đổi hai phần tử có thể vẫn khác loại. Phải dùng `elif` / `continue` để kiểm tra lại điều kiện sau mỗi bước.

In [24]:
def wrong_single_skip(s):
    chars = list(s)
    i, j = 0, len(chars) - 1
    while i < j:
        if not chars[i].islower():
            i += 1
        if not chars[j].islower():
            j -= 1
        chars[i], chars[j] = chars[j], chars[i]                   # SAI: hoán đổi dù chưa chắc cùng loại
        i += 1
        j -= 1
    return ''.join(chars)

s = "##ab"
print(wrong_single_skip(s), "(đúng phải là", Solution().reverseByType(s), ")")
assert Solution().reverseByType(s) == "##ba" and wrong_single_skip(s) == "#ba#"

#ba# (đúng phải là ##ba )


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đảo một loại ký tự, giữ vị trí các loại khác" -> tách theo loại, đảo, rồi điền lại bằng iterator hoặc hai con trỏ có bỏ qua ký tự khác loại.
- Hai nhóm độc lập -> làm lần lượt từng nhóm, thứ tự không quan trọng.
- `iter(reversed(list))` + `next()` là cách ngắn gọn nhất để "điền lần lượt".
- Hai con trỏ: `while i < j:` rồi `if ... elif ... else`, **không** để hai `if` độc lập.
- Test tự tạo: chuỗi toàn một loại (kết quả `s[::-1]`), chuỗi xen kẽ, chuỗi có ký tự đặc biệt ở hai đầu.
- Tính chất tự kiểm tra: loại ở mỗi vị trí không đổi; áp dụng hai lần trả về chuỗi gốc.

## 15. Biến thể của bài

1. **Reverse Only Letters** (LeetCode 917): chỉ đảo chữ cái, giữ nguyên ký tự khác tại chỗ (một nhóm).
2. **Reverse Vowels of a String** (345): chỉ đảo nguyên âm.
3. **Reverse String** (344): đảo cả chuỗi tại chỗ.
4. **Valid Palindrome** (125): bỏ qua ký tự không phải chữ/số bằng hai con trỏ.
5. **Reverse Words in a String** (151): đảo thứ tự các từ.
6. **Sort Vowels in a String** (2785): sắp xếp riêng nguyên âm, giữ vị trí phụ âm (cùng mô hình "ô có màu").

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| Brute force | thứ hạng đối xứng trong nhóm | `O(n²)` | `O(n)` | Oracle kiểm tra |
| Hai list + iterator | `iter(reversed(...))` + `next` | `O(n)` | `O(n)` | Nộp bài, ngắn gọn |
| Hai con trỏ từng nhóm | `reverse_group(pred)` hai lần | `O(n)` | `O(n)` | Mẫu "đảo tại chỗ" quen thuộc |
| Danh sách vị trí | điền ngược vào đúng vị trí | `O(n)` | `O(n)` | Hình dung "ô có màu" |
| Ngăn xếp | pop theo từng loại | `O(n)` | `O(n)` | Liên hệ LIFO |

Nộp cách hai list + iterator.

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().reverseByType("ab!cd")` và `Solution().reverseByType("(a)")` ra gì?

**Bài 2 (tính tay):** với `s = "x@y#z"`, liệt kê hai nhóm, đảo từng nhóm và ghép lại.

**Bài 3 (điền TODO):** hoàn thành `reverse_only_letters(s)` (LeetCode 917 dạng đơn giản): chỉ đảo chữ cái thường, các ký tự khác giữ nguyên tại chỗ.

**Bài 4 (tự viết lại):** viết `reverse_vowels(s)` (LeetCode 345, chỉ chữ thường): đảo các nguyên âm `aeiou` với nhau.

**Bài 5 (mini challenge):** viết `reverse_by_predicate(s, pred)` tổng quát: đảo các ký tự thỏa `pred` với nhau, giữ nguyên vị trí các ký tự còn lại. Dùng nó để biểu diễn cả `reverseByType` (hai lần gọi).

In [25]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().reverseByType("ab!cd"), Solution().reverseByType("(a)"))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def reverse_only_letters_todo(s):
    # TODO: đảo chữ cái, giữ nguyên ký tự khác (dùng iter(reversed(...)) hoặc hai con trỏ)
    return s

dc!ba )a(


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: `"ab!cd"` có chữ cái `a b c d` và đúng một ký tự đặc biệt. Bài 2: chữ cái `x y z`, đặc biệt `@ #`.

**Mức 2 (vừa):** Bài 3: giống nhóm "chữ cái" của Cách 2 nhưng ký tự khác `c` giữ nguyên. Bài 4: nhóm là các nguyên âm. Bài 5: nhận `pred` rồi dùng đúng khung hai con trỏ của Cách 3.

**Mức 3 (gần đáp án):** Bài 1: `"dc!ba"` và `")a("`. Bài 2: chữ cái đảo thành `z y x`, đặc biệt đảo thành `# @`: kết quả `"z#y@x"`. Bài 4: `reverse_vowels("hello") = "holle"`.

## 19. Đáp án tham khảo cho bài tập

In [26]:
# Bài 1, 2
assert Solution().reverseByType("ab!cd") == "dc!ba" and Solution().reverseByType("(a)") == ")a("
assert Solution().reverseByType("x@y#z") == "z#y@x"

# Bài 3
def reverse_only_letters(s):
    it = iter(reversed([c for c in s if c.islower()]))
    return ''.join(next(it) if c.islower() else c for c in s)
assert reverse_only_letters(")ebc#da@f(") == ")fad#cb@e(" and reverse_only_letters("a-bC") == "b-aC"

# Bài 4
def reverse_vowels(s):
    vowels = set("aeiou")
    it = iter(reversed([c for c in s if c in vowels]))
    return ''.join(next(it) if c in vowels else c for c in s)
assert reverse_vowels("hello") == "holle" and reverse_vowels("leetcode") == "leotcede" and reverse_vowels("xyz") == "xyz"

# Bài 5
def reverse_by_predicate(s, pred):
    chars = list(s)
    i, j = 0, len(chars) - 1
    while i < j:
        if not pred(chars[i]):
            i += 1
        elif not pred(chars[j]):
            j -= 1
        else:
            chars[i], chars[j] = chars[j], chars[i]
            i += 1
            j -= 1
    return ''.join(chars)
for t in (")ebc#da@f(", "z", "!@#$%^&*()", "a#b!c"):
    step = reverse_by_predicate(t, lambda c: c.islower())
    step = reverse_by_predicate(step, lambda c: not c.islower())
    assert step == Solution().reverseByType(t)
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đảo trong từng nhóm, giữ vị trí nhóm":**

```python
letters = iter(reversed([c for c in s if c.islower()]))
specials = iter(reversed([c for c in s if not c.islower()]))
return ''.join(next(letters) if c.islower() else next(specials) for c in s)
# hoặc hai con trỏ: while i < j: if không thuộc nhóm -> bỏ qua; elif ... ; else hoán đổi
```

**Checklist trước khi submit:**

- [ ] Đảo **cả hai** nhóm (chữ cái và ký tự đặc biệt), mỗi nhóm đảo **riêng**?
- [ ] Loại ký tự ở mỗi vị trí giữ nguyên (không dùng `s[::-1]` cho cả chuỗi)?
- [ ] Hai con trỏ dùng `if / elif / else`, kiểm tra cùng loại trước khi hoán đổi?
- [ ] Chuỗi bất biến: chuyển sang `list` hoặc dựng bằng `join`?
- [ ] Thử `"z"`, `"!@#$%^&*()"` và ví dụ `")ebc#da@f("`?
- [ ] Chữ ký đúng `reverseByType(self, s: str) -> str`?